In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')  # ensure UTC+8 awareness if needed later

# Load CSV files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# 1) Total row count for each file
row_counts = {
    'metric.csv': int(metric_df.shape[0]),
    'trace.csv': int(trace_df.shape[0]),
    'log.csv': int(log_df.shape[0]),
    'error_logs.csv': int(error_logs_df.shape[0])
}

# 2) Global min(timestamp) and max(timestamp) across all files (keep as raw numeric timestamps)
# Collect timestamp mins/maxs from files that have rows
mins = []
maxs = []
for df in (metric_df, trace_df, log_df, error_logs_df):
    if not df.empty and 'timestamp' in df.columns:
        # ensure numeric (some files might have non-numeric if corrupted; coerce)
        ts = pd.to_numeric(df['timestamp'], errors='coerce')
        if ts.notna().any():
            mins.append(int(ts.min()))
            maxs.append(int(ts.max()))

global_min_timestamp = int(min(mins)) if mins else None
global_max_timestamp = int(max(maxs)) if maxs else None

global_timestamp_range = {'min_timestamp': global_min_timestamp, 'max_timestamp': global_max_timestamp}

# 3) Unique cmdb_id values present across all files
cmdb_ids = set()
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'cmdb_id' in df.columns:
        cmdb_ids.update(df['cmdb_id'].dropna().unique().tolist())
unique_cmdb_ids = sorted(list(cmdb_ids))

# 4) For metric.csv: unique (cmdb_id, kpi_name) pairs with counts, top 20 by count
metric_pairs = (
    metric_df.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)
# convert counts to native python int for JSON-serializable output
metric_top20 = [
    {'cmdb_id': row['cmdb_id'], 'kpi_name': row['kpi_name'], 'count': int(row['count'])}
    for _, row in metric_pairs.iterrows()
]

# 5) For trace.csv: unique (cmdb_id, trace_name) pairs with counts, top 20 by count
trace_pairs = (
    trace_df.groupby(['cmdb_id', 'trace_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)
trace_top20 = [
    {'cmdb_id': row['cmdb_id'], 'trace_name': row['trace_name'], 'count': int(row['count'])}
    for _, row in trace_pairs.iterrows()
]

# 6) For log.csv: unique (cmdb_id, log_name) pairs with counts, top 20 by count
log_pairs = (
    log_df.groupby(['cmdb_id', 'log_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)
log_top20 = [
    {'cmdb_id': row['cmdb_id'], 'log_name': row['log_name'], 'count': int(row['count'])}
    for _, row in log_pairs.iterrows()
]

# Assemble final JSON-serializable summary
summary = {
    'row_counts': row_counts,
    'global_timestamp_range': global_timestamp_range,
    'unique_cmdb_ids': unique_cmdb_ids,
    'metric_top20_pairs': metric_top20,
    'trace_top20_pairs': trace_top20,
    'log_top20_pairs': log_top20
}

summary

```
Out[1]:
```
summary_text = (
    "Summary of telemetry files:\n"
    "1) Row counts: metric.csv=22525, trace.csv=7260, log.csv=489, error_logs.csv=0.\n"
    "2) Global timestamp range (raw seconds): min=1616491800, max=1616493600.\n"
    "3) Unique cmdb_id values (29): IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, "
    "ServiceTest1, ServiceTest10, ServiceTest11, ServiceTest2, ServiceTest3, ServiceTest4, ServiceTest5, "
    "ServiceTest6, ServiceTest7, ServiceTest8, ServiceTest9, Tomcat01, Tomcat02, Tomcat03, Tomcat04, "
    "apache01, apache02, dockerA1, dockerA2, dockerB1, dockerB2.\n"
    "4) metric.csv top (cmdb_id, kpi_name) pairs (top 20 by count):\n"
    "   - Tomcat04 | Tomcat-Requests_7441-\"http-nio-8003\"_ErrorCountRequestInfo | 31\n"
    "   - Tomcat04 | Tomcat-Requests_7441-\"http-nio-8003\"_MaxTimeRequestInfo | 31\n"
    "   - Tomcat04 | Tomcat-Requests_7441-\"http-nio-8003\"_ProcessingTimeRequestInfo | 31\n"
    "   - Tomcat04 | Tomcat-Requests_7441-\"http-nio-8003\"_RequestCountRequestInfo | 31\n"
    "   - Tomcat04 | Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter | 31\n"
    "   - Tomcat04 | Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCounter | 31\n"
    "   - Tomcat04 | OSLinux-OSLinux_ZABBIX_Host_Uptime | 31\n"
    "   - Tomcat04 | OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies | 31\n"
    "   - Tomcat04 | OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount | 31\n"
    "   - MG01 | JVM-Threads_7779_JVM_ThreadCount_Threads | 31\n"
    "   (plus 10 more entries; all shown in the earlier result, counts mostly 31).\n"
    "5) trace.csv top (cmdb_id, trace_name) pairs (top 20 by count):\n"
    "   - dockerB2 | trace.self.row_count | 31\n"
    "   - dockerA2 | trace.self.duration_mean | 31\n"
    "   - dockerA2 | trace.self.duration_p95 | 31\n"
    "   - dockerA2 | trace.self.row_count | 31\n"
    "   - dockerA2 | trace.from_MG01.duration_mean | 31\n"
    "   - dockerA2 | trace.from_MG01.duration_p95 | 31\n"
    "   - dockerA2 | trace.from_MG01.row_count | 31\n"
    "   - dockerA2 | trace.from_MG02.duration_mean | 31\n"
    "   - dockerA2 | trace.from_MG02.duration_p95 | 31\n"
    "   - dockerA2 | trace.from_MG02.row_count | 31\n"
    "   (plus 10 more entries; counts mostly 31).\n"
    "6) log.csv top (cmdb_id, log_name) pairs (top 20 by count):\n"
    "   - Tomcat02 | log.row_count | 31\n"
    "   - Tomcat02 | log.error_count | 31\n"
    "   - Tomcat01 | log.row_count | 31\n"
    "   - Tomcat01 | log.error_count | 31\n"
    "   - apache02 | log.error_count | 31\n"
    "   - apache01 | log.row_count | 31\n"
    "   - apache02 | log.row_count | 31\n"
    "   - apache01 | log.error_count | 31\n"
    "   - Tomcat03 | log.error_count | 31\n"
    "   - Tomcat04 | log.row_count | 31\n"
    "   (plus remaining entries including Mysql01 with 27 rows and some gc metrics).\n"
    "Note: All lists were limited to at most 20 entries per category in the earlier output."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

{'row_counts': {'metric.csv': 22525, 'trace.csv': 7260, 'log.csv': 489, 'error_logs.csv': 0}, 'global_timestamp_range': {'min_timestamp': 1616491800, 'max_timestamp': 1616493600}, 'unique_cmdb_ids': ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'metric_top20_pairs': [{'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCounter', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount', 'count': 31}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'count': 31}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'count': 31}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'count': 31}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'count': 31}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'count': 31}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'count': 31}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo', 'count': 31}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'count': 31}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONKeepaliveCounter', 'count': 31}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONRejectedSessions', 'count': 31}], 'trace_top20_pairs': [{'cmdb_id': 'dockerB2', 'trace_name': 'trace.self.row_count', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.self.duration_mean', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.self.duration_p95', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.self.row_count', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.from_MG01.duration_mean', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.from_MG01.duration_p95', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.from_MG01.row_count', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.from_MG02.duration_mean', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.from_MG02.duration_p95', 'count': 31}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.from_MG02.row_count', 'count': 31}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.self.duration_mean', 'count': 31}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.self.duration_p95', 'count': 31}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.self.row_count', 'count': 31}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.from_MG02.duration_mean', 'count': 31}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.from_MG02.duration_p95', 'count': 31}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.from_MG02.row_count', 'count': 31}, {'cmdb_id': 'MG01', 'trace_name': 'trace.self.duration_mean', 'count': 31}, {'cmdb_id': 'MG01', 'trace_name': 'trace.self.duration_p95', 'count': 31}, {'cmdb_id': 'MG01', 'trace_name': 'trace.from_dockerB1.duration_mean', 'count': 31}, {'cmdb_id': 'MG01', 'trace_name': 'trace.from_dockerB1.duration_p95', 'count': 31}], 'log_top20_pairs': [{'cmdb_id': 'Tomcat02', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'apache02', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'apache01', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'apache02', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'apache01', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.error_count', 'count': 31}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.row_count', 'count': 31}, {'cmdb_id': 'Mysql01', 'log_name': 'log.row_count', 'count': 27}, {'cmdb_id': 'Mysql01', 'log_name': 'log.error_count', 'count': 27}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.gc_pause_max', 'count': 16}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.gc_pause_max', 'count': 15}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.gc_pause_max', 'count': 15}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.gc_pause_max', 'count': 15}, {'cmdb_id': 'Mysql02', 'log_name': 'log.row_count', 'count': 1}, {'cmdb_id': 'Mysql02', 'log_name': 'log.error_count', 'count': 1}]}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df from previous state; reload if not present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')  # UTC+8 as required

# Allowed root-cause components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# Filter metric rows for allowed components (use full series)
metric_allowed = metric_df[metric_df['cmdb_id'].isin(allowed_components)].copy()

# Ensure 'value' is numeric
metric_allowed['value'] = pd.to_numeric(metric_allowed['value'], errors='coerce')

# Group and compute global stats per (cmdb_id, kpi_name)
grouped = (
    metric_allowed
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        count=('value', 'count'),
        p95=('value', lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan),
        p5 =('value', lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan),
        mean=('value', lambda x: float(x.mean()) if x.notna().any() else np.nan)
    )
    .reset_index()
)

# Total distinct pairs for these components
total_pairs = int(grouped.shape[0])

# Sort by count descending and limit to top 20
top20_df = grouped.sort_values('count', ascending=False).head(20)

# Build JSON-serializable list of dicts with native Python types and NaN->None
def to_native(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer, np.int64)):
        return int(v)
    if isinstance(v, (np.floating, np.float64, float)):
        return float(v)
    return v

result_list = []
for _, r in top20_df.iterrows():
    result_list.append({
        'cmdb_id': r['cmdb_id'],
        'kpi_name': r['kpi_name'],
        'count': to_native(r['count']),
        'p95': to_native(r['p95']),
        'p5': to_native(r['p5']),
        'mean': to_native(r['mean'])
    })

# Display compact outputs: the list (<=20) and the total count of distinct pairs
result_list, total_pairs

```
Out[2]:
```
Summary of results (computed on the full metric series for the specified components):

- Total distinct (cmdb_id, kpi_name) pairs found: 1111.

Top 20 pairs by data-point count (each: cmdb_id | kpi_name | count, p95, p5, mean):
1) Tomcat04 | Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo | count=31, p95=500.0, p5=500.0, mean=500.0  
2) Tomcat04 | OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies | count=31, p95=1.0, p5=1.0, mean=1.0  
3) Tomcat04 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT | count=31, p95=0.0, p5=0.0, mean=0.0  
4) Tomcat04 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT | count=31, p95=0.0, p5=0.0, mean=0.0  
5) MG01 | JVM-Threads_7779_JVM_ThreadCount_Threads | count=31, p95=76.0, p5=68.5, mean=72.48387096774194  
6) MG01 | JVM-Memory_7779_JVM_Memory_HeapMemoryMax | count=31, p95=2040135680.0, p5=2040135680.0, mean=2040135680.0  
7) MG01 | JVM-Memory_7779_JVM_Memory_HeapMemoryUsed | count=31, p95=971017156.0, p5=207865532.0, mean=551172606.7096775  
8) MG01 | JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed | count=31, p95=59071496.0, p5=59071496.0, mean=59071496.0  
9) Tomcat03 | Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONActiveCounter | count=31, p95=0.0, p5=0.0, mean=0.0  
10) Tomcat03 | Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONKeepaliveCounter | count=31, p95=0.0, p5=0.0, mean=0.0  
11) Tomcat03 | Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONRejectedSessions | count=31, p95=0.0, p5=0.0, mean=0.0  
12) Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo | count=31, p95=100.0, p5=100.0, mean=100.0  
13) Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo | count=31, p95=4.0, p5=0.0, mean=1.7096774193548387  
14) Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo | count=31, p95=500.0, p5=500.0, mean=500.0  
15) Tomcat03 | Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCounter | count=31, p95=0.0, p5=0.0, mean=0.0  
16) Tomcat03 | Tomcat-Sessions_7441--UOCP_SESSIONRejectedSessions | count=31, p95=0.0, p5=0.0, mean=0.0  
17) Tomcat03 | Tomcat-Sessions_7441--_SESSIONActiveCounter | count=31, p95=0.0, p5=0.0, mean=0.0  
18) Tomcat03 | Tomcat-Sessions_7441--_SESSIONKeepaliveCounter | count=31, p95=0.0, p5=0.0, mean=0.0  
19) Tomcat03 | Tomcat-Sessions_7441--_SESSIONRejectedSessions | count=31, p95=0.0, p5=0.0, mean=0.0  
20) Tomcat03 | Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONActiveCounter | count=31, p95=0.0, p5=0.0, mean=0.0

If you want the full JSON-serializable list (up to 20 entries) or to adjust sorting/filters, I can produce that next.

The original code execution output of IPython Kernel is also provided below for reference:

([{'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'count': 31, 'p95': 500.0, 'p5': 500.0, 'mean': 500.0}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 'count': 31, 'p95': 1.0, 'p5': 1.0, 'mean': 1.0}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'count': 31, 'p95': 76.0, 'p5': 68.5, 'mean': 72.48387096774194}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'count': 31, 'p95': 2040135680.0, 'p5': 2040135680.0, 'mean': 2040135680.0}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'count': 31, 'p95': 971017156.0, 'p5': 207865532.0, 'mean': 551172606.7096775}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'count': 31, 'p95': 59071496.0, 'p5': 59071496.0, 'mean': 59071496.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONActiveCounter', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONKeepaliveCounter', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SESSIONRejectedSessions', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'count': 31, 'p95': 100.0, 'p5': 100.0, 'mean': 100.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo', 'count': 31, 'p95': 4.0, 'p5': 0.0, 'mean': 1.7096774193548387}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'count': 31, 'p95': 500.0, 'p5': 500.0, 'mean': 500.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCounter', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--UOCP_SESSIONRejectedSessions', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONActiveCounter', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONKeepaliveCounter', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--_SESSIONRejectedSessions', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SESSIONActiveCounter', 'count': 31, 'p95': 0.0, 'p5': 0.0, 'mean': 0.0}], 1111)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')  # UTC+8

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Allowed root-cause components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# Ensure 'value' numeric in metric_df
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Reuse precomputed thresholds if present (grouped), else compute global thresholds using full series
try:
    thresholds = grouped.copy()  # from previous cell: grouped contains count,p95,p5,mean
except NameError:
    metric_allowed_full = metric_df[metric_df['cmdb_id'].isin(allowed_components)].copy()
    thresholds = (
        metric_allowed_full
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(count='count',
             p95=lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan,
             p5 =lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan,
             mean=lambda x: float(x.mean()) if x.notna().any() else np.nan)
        .reset_index()
    )

# Drop thresholds where p95 or p5 are NaN (can't evaluate anomalies)
thresholds = thresholds.dropna(subset=['p95','p5']).copy()

# Filter metric rows to the given time window and allowed components
start_ts = 1616491800
end_ts   = 1616493600
metric_window = metric_df[
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['cmdb_id'].isin(allowed_components))
].copy()

# Merge windowed metrics with thresholds to evaluate anomalies only for known (cmdb_id,kpi_name) pairs
merged = pd.merge(
    metric_window,
    thresholds[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='inner'
)

# Identify anomalies: value >= p95 OR value <= p5
merged['is_anomaly'] = ((merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])) & merged['value'].notna()

# Filter anomalous rows
anomalies = merged[merged['is_anomaly']].copy()

# Aggregate per (cmdb_id, kpi_name)
if not anomalies.empty:
    agg = (
        anomalies
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly','sum'),
            earliest_anomaly_timestamp=('timestamp','min'),
            latest_anomaly_timestamp=('timestamp','max'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            global_p95=('p95','first'),
            global_p5 =('p5','first')
        )
    )
else:
    # empty dataframe with expected columns
    agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp',
        'latest_anomaly_timestamp','max_value_in_window','min_value_in_window',
        'global_p95','global_p5'
    ])

# Total number of (cmdb_id,kpi_name) pairs that have at least one anomaly
total_anomalous_pairs = int(agg.shape[0])

# Top 20 pairs by anomaly_count desc
agg_top20 = agg.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Prepare JSON-serializable report (convert numpy types to native python)
def to_native(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v

top20_list = []
for _, r in agg_top20.iterrows():
    top20_list.append({
        'cmdb_id': r['cmdb_id'],
        'kpi_name': r['kpi_name'],
        'anomaly_count': to_native(r['anomaly_count']),
        'earliest_anomaly_timestamp': to_native(r['earliest_anomaly_timestamp']),
        'latest_anomaly_timestamp': to_native(r['latest_anomaly_timestamp']),
        'max_value_in_window': to_native(r['max_value_in_window']),
        'min_value_in_window': to_native(r['min_value_in_window']),
        'global_p95': to_native(r['global_p95']),
        'global_p5': to_native(r['global_p5'])
    })

report = {
    'total_anomalous_pairs': total_anomalous_pairs,
    'top20': top20_list
}

# Display compact outputs: the JSON-serializable report and a compact DataFrame view for inspection
report, agg_top20[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','latest_anomaly_timestamp']].head(20)

```
Out[3]:
```
Summary of metric anomalies in window [1616491800, 1616493600] using global thresholds:

- Total distinct (cmdb_id, kpi_name) pairs with ≥1 anomaly: 1111.

Top 20 pairs by anomaly count (fields: cmdb_id | kpi_name | anomaly_count | earliest_ts | latest_ts | max_value | min_value | global_p95 | global_p5):

1) Tomcat04 | OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount | 31 | 1616491800 | 1616493600 | 3.0 | 3.0 | 3.0 | 3.0  
2) IG02 | JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed | 31 | 1616491800 | 1616493600 | 58541144.0 | 58541144.0 | 58541144.0 | 58541144.0  
3) Mysql01 | Mysql-MySQL_3306_Innodb buffer pool read ahead evicted | 31 | 1616491800 | 1616493600 | 0.0 | 0.0 | 0.0 | 0.0  
4) Mysql01 | Mysql-MySQL_3306_Innodb buffer pool read ahead rnd | 31 | 1616491800 | 1616493600 | 0.0 | 0.0 | 0.0 | 0.0  
5) Mysql01 | Mysql-MySQL_3306_Innodb buffer pool reads | 31 | 1616491800 | 1616493600 | 273.0 | 0.0 | 273.0 | 0.0  
6) Mysql01 | Mysql-MySQL_3306_Innodb buffer pool wait free | 31 | 1616491800 | 1616493600 | 0.0 | 0.0 | 0.0 | 0.0  
7) Mysql01 | Mysql-MySQL_3306_Handler Savepoint Rollback | 31 | 1616491800 | 1616493600 | 0.0 | 0.0 | 0.0 | 0.0  
8) Mysql01 | Mysql-MySQL_3306_Aborted Connects | 31 | 1616491800 | 1616493600 | 37916.0 | 0.0 | 37916.0 | 0.0  
9) Mysql01 | Mysql-MySQL_3306_Binlog Cache Disk Use | 31 | 1616491800 | 1616493600 | 59.0 | 0.0 | 59.0 | 0.0  
10) MG02 | OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies | 31 | 1616491800 | 1616493600 | 1.0 | 1.0 | 1.0 | 1.0  
11) Mysql01 | Mysql-MySQL_3306_Handler Read Next | 31 | 1616491800 | 1616493600 | 12333892.0 | 0.0 | 12333892.0 | 0.0  
12) Mysql01 | Mysql-MySQL_3306_Handler Read Prev | 31 | 1616491800 | 1616493600 | 722.0 | 0.0 | 722.0 | 0.0  
13) Mysql01 | Mysql-MySQL_3306_Handler Read Rnd | 31 | 1616491800 | 1616493600 | 1264745.0 | 0.0 | 1264745.0 | 0.0  
14) Mysql01 | Mysql-MySQL_3306_Handler Rollback | 31 | 1616491800 | 1616493600 | 187.0 | 0.0 | 187.0 | 0.0  
15) Mysql01 | Mysql-MySQL_3306_Handler Savepoint | 31 | 1616491800 | 1616493600 | 0.0 | 0.0 | 0.0 | 0.0  
16) Mysql01 | Mysql-MySQL_3306_Innodb buffer pool pages misc | 31 | 1616491800 | 1616493600 | 117.0 | 0.0 | 117.0 | 0.0  
17) Mysql01 | Mysql-MySQL_3306_Innodb buffer pool pages total | 31 | 1616491800 | 1616493600 | 262112.0 | 0.0 | 262112.0 | 0.0  
18) Mysql01 | Mysql-MySQL_3306_Innodb buffer pool read ahead | 31 | 1616491800 | 1616493600 | 0.0 | 0.0 | 0.0 | 0.0  
19) Mysql01 | Mysql-MySQL_3306_Key Read Requests | 31 | 1616491800 | 1616493600 | 5177.0 | 0.0 | 5177.0 | 0.0  
20) Mysql01 | Mysql-MySQL_3306_Key Reads | 31 | 1616491800 | 1616493600 | 12.0 | 0.0 | 12.0 | 0.0

All top-20 pairs shown have anomalies at every timestamp in the window (anomaly_count = 31). If you want the full list or CSV-style JSON for all anomalous pairs, I can provide it.

The original code execution output of IPython Kernel is also provided below for reference:

({'total_anomalous_pairs': 1111, 'top20': [{'cmdb_id': 'Tomcat04', 'kpi_name': 'OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 3.0, 'min_value_in_window': 3.0, 'global_p95': 3.0, 'global_p5': 3.0}, {'cmdb_id': 'IG02', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 58541144.0, 'min_value_in_window': 58541144.0, 'global_p95': 58541144.0, 'global_p5': 58541144.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Innodb buffer pool read ahead evicted', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'global_p95': 0.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Innodb buffer pool read ahead rnd', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'global_p95': 0.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Innodb buffer pool reads', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 273.0, 'min_value_in_window': 0.0, 'global_p95': 273.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Innodb buffer pool wait free', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'global_p95': 0.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Handler Savepoint Rollback', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'global_p95': 0.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Aborted Connects', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 37916.0, 'min_value_in_window': 0.0, 'global_p95': 37916.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Binlog Cache Disk Use', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 59.0, 'min_value_in_window': 0.0, 'global_p95': 59.0, 'global_p5': 0.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 1.0, 'min_value_in_window': 1.0, 'global_p95': 1.0, 'global_p5': 1.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Handler Read Next', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 12333892.0, 'min_value_in_window': 0.0, 'global_p95': 12333892.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Handler Read Prev', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 722.0, 'min_value_in_window': 0.0, 'global_p95': 722.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Handler Read Rnd', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 1264745.0, 'min_value_in_window': 0.0, 'global_p95': 1264745.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Handler Rollback', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 187.0, 'min_value_in_window': 0.0, 'global_p95': 187.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Handler Savepoint', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'global_p95': 0.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Innodb buffer pool pages misc', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 117.0, 'min_value_in_window': 0.0, 'global_p95': 117.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Innodb buffer pool pages total', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 262112.0, 'min_value_in_window': 0.0, 'global_p95': 262112.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Innodb buffer pool read ahead', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 0.0, 'min_value_in_window': 0.0, 'global_p95': 0.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Key Read Requests', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 5177.0, 'min_value_in_window': 0.0, 'global_p95': 5177.0, 'global_p5': 0.0}, {'cmdb_id': 'Mysql01', 'kpi_name': 'Mysql-MySQL_3306_Key Reads', 'anomaly_count': 31, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 12.0, 'min_value_in_window': 0.0, 'global_p95': 12.0, 'global_p5': 0.0}]},      cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp  latest_anomaly_timestamp
0   Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31                  1616491800                1616493600
1       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31                  1616491800                1616493600
2    Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...             31                  1616491800                1616493600
3    Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...             31                  1616491800                1616493600
4    Mysql01          Mysql-MySQL_3306_Innodb buffer pool reads             31                  1616491800                1616493600
5    Mysql01      Mysql-MySQL_3306_Innodb buffer pool wait free             31                  1616491800                1616493600
6    Mysql01        Mysql-MySQL_3306_Handler Savepoint Rollback             31                  1616491800                1616493600
7    Mysql01                  Mysql-MySQL_3306_Aborted Connects             31                  1616491800                1616493600
8    Mysql01             Mysql-MySQL_3306_Binlog Cache Disk Use             31                  1616491800                1616493600
9       MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31                  1616491800                1616493600
10   Mysql01                 Mysql-MySQL_3306_Handler Read Next             31                  1616491800                1616493600
11   Mysql01                 Mysql-MySQL_3306_Handler Read Prev             31                  1616491800                1616493600
12   Mysql01                  Mysql-MySQL_3306_Handler Read Rnd             31                  1616491800                1616493600
13   Mysql01                  Mysql-MySQL_3306_Handler Rollback             31                  1616491800                1616493600
14   Mysql01                 Mysql-MySQL_3306_Handler Savepoint             31                  1616491800                1616493600
15   Mysql01     Mysql-MySQL_3306_Innodb buffer pool pages misc             31                  1616491800                1616493600
16   Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages total             31                  1616491800                1616493600
17   Mysql01     Mysql-MySQL_3306_Innodb buffer pool read ahead             31                  1616491800                1616493600
18   Mysql01                 Mysql-MySQL_3306_Key Read Requests             31                  1616491800                1616493600
19   Mysql01                         Mysql-MySQL_3306_Key Reads             31                  1616491800                1616493600)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables if present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure numeric value
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Allowed components (reuse)
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# Reuse thresholds (global p95/p5) computed earlier; if missing, compute now using full series (rule 9)
try:
    thresholds
except NameError:
    metric_allowed_full = metric_df[metric_df['cmdb_id'].isin(allowed_components)].copy()
    thresholds = (
        metric_allowed_full
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(
            count='count',
            p95=lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan,
            p5 =lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan,
            mean=lambda x: float(x.mean()) if x.notna().any() else np.nan
        )
        .reset_index()
    )
thresholds = thresholds.dropna(subset=['p95','p5']).copy()

# Define incident window
start_ts = 1616491800
end_ts   = 1616493600

# Windowed metric rows for allowed components
metric_window = metric_df[
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['cmdb_id'].isin(allowed_components))
].copy()

# Merge with thresholds to evaluate anomalies for only known pairs
merged = pd.merge(
    metric_window,
    thresholds[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='inner'
)

# Identify anomalies
merged['is_anomaly'] = ((merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])) & merged['value'].notna()

# Keep only anomalous rows
anomalies = merged[merged['is_anomaly']].copy()

# If no anomalies, return empty structured report
if anomalies.empty:
    report = {'per_cmdb': [], 'JVM_OOM_candidates': []}
    report, pd.DataFrame(columns=['cmdb_id','earliest_ts_where_ratio_ge_0_9','max_ratio'])
else:
    # Keyword->reason mapping (case-insensitive)
    def classify_kpi(k):
        k_low = k.lower()
        reasons = set()
        # CPU
        if ('cpu' in k_low) or ('load' in k_low):
            reasons.add("high CPU usage")
        # Memory
        if ('memory' in k_low) or ('heapmemoryused' in k_low) or ('heapmemory' in k_low) or ('heapmemorymax' in k_low) or ('heap' in k_low) or ('swap' in k_low):
            reasons.add("high memory usage")
        # Network
        if ('network' in k_low) or ('tcp' in k_low) or ('udp' in k_low):
            reasons.add("network latency")
        if ('packet' in k_low):
            reasons.add("network packet loss")
        # Disk / IO
        if (' io ' in k_low) or k_low.endswith(' io') or ('io' in k_low) or ('diskio' in k_low) or ('read' in k_low):
            # prefer disk I/O read usage for 'io' or 'read'
            if ('io' in k_low) or ('read' in k_low):
                reasons.add("high disk I/O read usage")
        if ('space' in k_low) or ('filesystem' in k_low) or (' df' in k_low) or k_low.startswith('df') or ('df_' in k_low):
            reasons.add("high disk space usage")
        # JVM / threads / gc
        if ('jvm' in k_low) or ('gc' in k_low) or ('gc_pause' in k_low) or ('thread' in k_low) or ('threadcount' in k_low):
            reasons.add("high JVM CPU load")
        # swap already mapped above to memory
        return sorted(list(reasons))
    
    # Apply classification
    anomalies['reasons'] = anomalies['kpi_name'].apply(classify_kpi)
    # Expand to one row per reason for aggregation
    anomalies_exploded = anomalies.explode('reasons')
    # Remove rows with no matched reason (we only want rows that matched at least one keyword)
    anomalies_exploded = anomalies_exploded[anomalies_exploded['reasons'].notna() & (anomalies_exploded['reasons'] != '')].copy()
    
    # Aggregate per cmdb_id overall
    per_cmdb_total = (
        anomalies.groupby('cmdb_id', as_index=False)
        .agg(
            total_anomalies=('is_anomaly','sum'),
            distinct_anomalous_kpis=('kpi_name', lambda x: int(x.nunique()))
        )
    )
    # Per-cmdb per-reason aggregation
    per_reason = (
        anomalies_exploded.groupby(['cmdb_id','reasons'], as_index=False)
        .agg(
            anomaly_count_for_reason=('is_anomaly','sum'),
            earliest_anomaly_timestamp_for_reason=('timestamp','min'),
            latest_anomaly_timestamp_for_reason=('timestamp','max'),
            sample_kpi_names=('kpi_name', lambda x: list(pd.Series(x.unique()).tolist()[:5]))
        )
    )
    # Combine: build per-cmdb structured objects
    # Merge totals and then attach reason lists
    per_cmdb = per_cmdb_total.merge(per_cmdb_total[['cmdb_id']], on='cmdb_id')  # just to ensure DataFrame
    # prepare dict entries
    cmdb_entries = []
    # Order cmdbs by total_anomalies desc and limit to top 20
    top_cmdbs = per_cmdb_total.sort_values('total_anomalies', ascending=False).head(20)['cmdb_id'].tolist()
    for cmdb in top_cmdbs:
        tot_row = per_cmdb_total[per_cmdb_total['cmdb_id']==cmdb].iloc[0]
        reasons_rows = per_reason[per_reason['cmdb_id']==cmdb].sort_values('anomaly_count_for_reason', ascending=False)
        reasons_list = []
        for _, rr in reasons_rows.iterrows():
            reasons_list.append({
                'reason_name': rr['reasons'],
                'anomaly_count_for_reason': int(rr['anomaly_count_for_reason']),
                'earliest_anomaly_timestamp_for_reason': int(rr['earliest_anomaly_timestamp_for_reason']),
                'latest_anomaly_timestamp_for_reason': int(rr['latest_anomaly_timestamp_for_reason']),
                'sample_kpi_names': rr['sample_kpi_names']
            })
        cmdb_entries.append({
            'cmdb_id': cmdb,
            'total_anomalies': int(tot_row['total_anomalies']),
            'distinct_anomalous_kpis': int(tot_row['distinct_anomalous_kpis']),
            'reasons': reasons_list
        })
    
    # JVM OOM candidates:
    # For each cmdb_id, find kpis containing HeapMemoryUsed and HeapMemoryMax (case-insensitive), within the window,
    # align by timestamp, compute used/max ratio, and check max ratio >= 0.9
    window_allowed = metric_window.copy()
    window_allowed['kpi_lower'] = window_allowed['kpi_name'].str.lower()
    # find candidates where both kpi types exist
    oom_candidates = []
    for cmdb in window_allowed['cmdb_id'].unique():
        sub = window_allowed[window_allowed['cmdb_id']==cmdb]
        used_mask = sub['kpi_lower'].str.contains('heapmemoryused', na=False)
        max_mask  = sub['kpi_lower'].str.contains('heapmemorymax', na=False)
        if used_mask.any() and max_mask.any():
            used = sub[used_mask][['timestamp','value']].rename(columns={'value':'used'})
            maxv = sub[max_mask][['timestamp','value']].rename(columns={'value':'max'})
            # align by timestamp: merge on timestamp; there might be multiple kpi entries per timestamp; aggregate by taking mean per timestamp
            used_agg = used.groupby('timestamp', as_index=False).agg(used=('used','mean'))
            max_agg  = maxv.groupby('timestamp', as_index=False).agg(max=('max','mean'))
            joined = pd.merge(used_agg, max_agg, on='timestamp', how='inner')
            # only consider positive max to avoid division by zero
            joined = joined[joined['max'] > 0].copy()
            if not joined.empty:
                joined['ratio'] = joined['used'] / joined['max']
                max_ratio = float(joined['ratio'].max())
                if max_ratio >= 0.9:
                    # earliest timestamp where ratio >=0.9
                    earliest_ts = int(joined[joined['ratio'] >= 0.9]['timestamp'].min())
                    oom_candidates.append({
                        'cmdb_id': cmdb,
                        'earliest_timestamp_ratio_ge_0_9': earliest_ts,
                        'max_ratio_observed': max_ratio
                    })
    # limit candidates to top 20 (by max_ratio desc)
    if oom_candidates:
        oom_df = pd.DataFrame(oom_candidates).sort_values('max_ratio_observed', ascending=False).head(20)
        oom_list = oom_df.to_dict('records')
    else:
        oom_list = []
    
    report = {
        'per_cmdb': cmdb_entries,
        'JVM_OOM_candidates': oom_list
    }

# Display compact results: report (JSON-serializable) and a small DataFrame view of JVM_OOM_candidates
report, pd.DataFrame(report['JVM_OOM_candidates']).head(20)

```
Out[4]:
```
Summary of anomalous metric rows in window [1616491800, 1616493600] (anomaly = value >= global_p95 OR value <= global_p5):

Overall:
- We analyzed allowed components and found per-cmdb anomaly summaries for the top components below.
- No JVM OOM candidates were detected (no cmdb_id had HeapMemoryUsed/HeapMemoryMax ratio ≥ 0.9 in the window).

Top cmdb_id summaries (sorted by total anomalous rows):

1) Mysql01 — total_anomalies: 2846; distinct_anomalous_kpis: 137.
   - Main reason categories (counts): high disk I/O read usage (694), high JVM CPU load (155), high memory usage (86), high CPU usage (41).

2) Mysql02 — total_anomalies: 2642; distinct_anomalous_kpis: 137.
   - Main reasons: high disk I/O read usage (636), high JVM CPU load (140), high memory usage (84), high CPU usage (42).

3) Tomcat04 — total_anomalies: 1428; distinct_anomalous_kpis: 122.
   - Main reasons: high disk I/O read usage (547), high disk space usage (352), network latency (117), high memory usage (100), high JVM CPU load (88), high CPU usage (80). Also some network packet loss (4).

4) Tomcat03 — total_anomalies: 1389; distinct_anomalous_kpis: 121.
   - Main reasons: high disk I/O read usage (550), high disk space usage (326), network latency (109), high memory usage (102), high JVM CPU load (88), high CPU usage (76). Some network packet loss (4).

5) Tomcat02 — total_anomalies: 1369; distinct_anomalous_kpis: 121.
   - Main reasons: high disk I/O read usage (550), high disk space usage (326), network latency (113), high JVM CPU load (88), high CPU usage (82), high memory usage (79). Some network packet loss (4).

6) Tomcat01 — total_anomalies: 1318; distinct_anomalous_kpis: 118.
   - Main reasons: high disk I/O read usage (456), high disk space usage (341), network latency (107), high memory usage (104), high JVM CPU load (89), high CPU usage (73). Some network packet loss (4).

7) MG01 — total_anomalies: 927; distinct_anomalous_kpis: 106.
   - Main reasons: high disk space usage (295), high memory usage (159), network latency (110), high CPU usage (95), high JVM CPU load (78), high disk I/O read usage (41). Some network packet loss (4).

8) Redis02 — total_anomalies: 917; distinct_anomalous_kpis: 76.
   - Main reasons: high disk space usage (334), high memory usage (117), network latency (69), high CPU usage (63), high disk I/O read usage (48).

9) MG02 — total_anomalies: 916; distinct_anomalous_kpis: 104.
   - Main reasons: high disk space usage (331), high memory usage (120), network latency (108), high CPU usage (94), high JVM CPU load (53), high disk I/O read usage (36). Some network packet loss (4).

10) IG02 — total_anomalies: 165; distinct_anomalous_kpis: 18.
    - Main reasons: high memory usage (97), high JVM CPU load (82), network latency (22), high CPU usage (14), high disk I/O read usage (14).

11) IG01 — total_anomalies: 151; distinct_anomalous_kpis: 18.
    - Main reasons: high memory usage (89), high JVM CPU load (80), network latency (19), high disk I/O read usage (13), high CPU usage (10).

12) Redis01 — total_anomalies: 97; distinct_anomalous_kpis: 11.
    - Main reasons: high memory usage (30), high disk I/O read usage (16), network latency (16), high CPU usage (11).

13) apache02 — total_anomalies: 70; distinct_anomalous_kpis: 11.
    - Main reasons: high memory usage (32), high CPU usage (13), high disk I/O read usage (6).

14) apache01 — total_anomalies: 64; distinct_anomalous_kpis: 11.
    - Main reasons: high memory usage (22), high CPU usage (17), high disk I/O read usage (5).

Notes:
- Counts shown are numbers of anomalous datapoints in the window mapped to each reason category (a single kpi row can map to multiple reasons if multiple keywords match).
- Sample kpi_name examples for each reason (and earliest/latest timestamps) were computed but not listed in full here; they are available in the structured report produced earlier.
- JVM_OOM_candidates is empty: no cmdb_id met the HeapMemoryUsed/HeapMemoryMax ≥ 0.9 condition within the window.

The original code execution output of IPython Kernel is also provided below for reference:

({'per_cmdb': [{'cmdb_id': 'Mysql01', 'total_anomalies': 2846, 'distinct_anomalous_kpis': 137, 'reasons': [{'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 694, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Handler Read Next', 'Mysql-MySQL_3306_Handler Read Prev', 'Mysql-MySQL_3306_Handler Read Rnd', 'Mysql-MySQL_3306_Innodb buffer pool read ahead', 'Mysql-MySQL_3306_Innodb buffer pool read ahead evicted']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 155, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Slow launch threads', 'Mysql-MySQL_3306_Threads Cached', 'Mysql-MySQL_3306_Threads Created', 'Mysql-MySQL_3306_ThreadsConnected', 'Mysql-MySQL_3306_ThreadsRunning']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 86, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Qcache Free Memory', 'Mysql-MySQL_3306_max trx lock memory bytes', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 41, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Com Load', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuUtil']}]}, {'cmdb_id': 'Mysql02', 'total_anomalies': 2642, 'distinct_anomalous_kpis': 137, 'reasons': [{'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 636, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Connections', 'Mysql-MySQL_3306_Handler Read First', 'Mysql-MySQL_3306_Handler Read Next', 'Mysql-MySQL_3306_Handler Read Rnd', 'Mysql-MySQL_3306_Handler Read Rnd Next']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 140, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Slow launch threads', 'Mysql-MySQL_3306_Threads Cached', 'Mysql-MySQL_3306_Threads Created', 'Mysql-MySQL_3306_ThreadsConnected', 'Mysql-MySQL_3306_ThreadsRunning']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 84, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Qcache Free Memory', 'Mysql-MySQL_3306_max trx lock memory bytes', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 42, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Mysql-MySQL_3306_Com Load', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil']}]}, {'cmdb_id': 'Tomcat04', 'total_anomalies': 1428, 'distinct_anomalous_kpis': 122, 'reasons': [{'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 547, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo', 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter']}, {'reason_name': 'high disk space usage', 'anomaly_count_for_reason': 352, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 117, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil', 'OSLinux-OSLinux_NETWORK_ens160_NETInErr', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 100, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent', 'OSLinux-OSLinux_SWAP_SWAP_Si', 'OSLinux-OSLinux_SWAP_SWAP_So', 'Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 88, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory', 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 80, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuUtil']}, {'reason_name': 'network packet loss', 'anomaly_count_for_reason': 4, 'earliest_anomaly_timestamp_for_reason': 1616492580, 'latest_anomaly_timestamp_for_reason': 1616493120, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut']}]}, {'cmdb_id': 'Tomcat03', 'total_anomalies': 1389, 'distinct_anomalous_kpis': 121, 'reasons': [{'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 550, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo', 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter']}, {'reason_name': 'high disk space usage', 'anomaly_count_for_reason': 326, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 109, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum', 'OSLinux-OSLinux_NETWORK_ens160_NETInErr', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 102, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent', 'OSLinux-OSLinux_SWAP_SWAP_Si', 'OSLinux-OSLinux_SWAP_SWAP_So', 'OSLinux-OSLinux_MEMORY_MEMORY_CacheMem']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 88, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory', 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 76, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil']}, {'reason_name': 'network packet loss', 'anomaly_count_for_reason': 4, 'earliest_anomaly_timestamp_for_reason': 1616493300, 'latest_anomaly_timestamp_for_reason': 1616493420, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut']}]}, {'cmdb_id': 'Tomcat02', 'total_anomalies': 1369, 'distinct_anomalous_kpis': 121, 'reasons': [{'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 550, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo', 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter']}, {'reason_name': 'high disk space usage', 'anomaly_count_for_reason': 326, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 113, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc', 'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum', 'OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 88, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 82, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuidle']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 79, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_UserMem', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent', 'OSLinux-OSLinux_SWAP_SWAP_Si', 'OSLinux-OSLinux_SWAP_SWAP_So']}, {'reason_name': 'network packet loss', 'anomaly_count_for_reason': 4, 'earliest_anomaly_timestamp_for_reason': 1616492580, 'latest_anomaly_timestamp_for_reason': 1616493480, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut']}]}, {'cmdb_id': 'Tomcat01', 'total_anomalies': 1318, 'distinct_anomalous_kpis': 118, 'reasons': [{'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 456, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_ProcessingTimeRequestInfo', 'Tomcat-Requests_7441-"http-nio-8003"_RequestCountRequestInfo', 'Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter']}, {'reason_name': 'high disk space usage', 'anomaly_count_for_reason': 341, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 107, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum', 'OSLinux-OSLinux_NETWORK_ens160_NETInErr', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 104, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent', 'OSLinux-OSLinux_SWAP_SWAP_Si', 'OSLinux-OSLinux_SWAP_SWAP_So', 'Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 89, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadCountThreadInfo', 'Tomcat-Threads_7441-"http-nio-8003"_MaxThreadsThreadInfo', 'Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory', 'Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory', 'Tomcat-Threads_7441-"http-nio-8003"_CurrentThreadsBusyThreadInfo']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 73, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime']}, {'reason_name': 'network packet loss', 'anomaly_count_for_reason': 4, 'earliest_anomaly_timestamp_for_reason': 1616492580, 'latest_anomaly_timestamp_for_reason': 1616493420, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut']}]}, {'cmdb_id': 'MG01', 'total_anomalies': 927, 'distinct_anomalous_kpis': 106, 'reasons': [{'reason_name': 'high disk space usage', 'anomaly_count_for_reason': 295, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 159, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_UserMem', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 110, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc', 'OSLinux-OSLinux_NETWORK_ens160_NETInErr']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 95, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU_CPULoad']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 78, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'JVM-Runtime_7779_JVM_JVM_Uptime', 'JVM-Operating System_7779_JVM_JVM_CPULoad']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 41, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Threads_7779_JVM_ThreadCount_Threads', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead', 'OSLinux-CPU_CPU_CPUWio']}, {'reason_name': 'network packet loss', 'anomaly_count_for_reason': 4, 'earliest_anomaly_timestamp_for_reason': 1616492100, 'latest_anomaly_timestamp_for_reason': 1616493480, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut']}]}, {'cmdb_id': 'Redis02', 'total_anomalies': 917, 'distinct_anomalous_kpis': 76, 'reasons': [{'reason_name': 'high disk space usage', 'anomaly_count_for_reason': 334, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-app_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-app_FSCapacity']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 117, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_CacheMem', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_UserMem', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 69, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 63, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPUSysTime']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 48, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-CPU_CPU_CPUWio']}]}, {'cmdb_id': 'MG02', 'total_anomalies': 916, 'distinct_anomalous_kpis': 104, 'reasons': [{'reason_name': 'high disk space usage', 'anomaly_count_for_reason': 331, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity']}, {'reason_name': 'high memory usage', 'anomaly_count_for_reason': 120, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 108, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 94, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'JVM-Operating System_7779_JVM_JVM_CPULoad', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUUserTime']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 53, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Runtime_7779_JVM_JVM_Uptime', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'JVM-Operating System_7779_JVM_JVM_CPULoad']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 36, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Threads_7779_JVM_ThreadCount_Threads', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite']}, {'reason_name': 'network packet loss', 'anomaly_count_for_reason': 4, 'earliest_anomaly_timestamp_for_reason': 1616492580, 'latest_anomaly_timestamp_for_reason': 1616493480, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn', 'OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut']}]}, {'cmdb_id': 'IG02', 'total_anomalies': 165, 'distinct_anomalous_kpis': 18, 'reasons': [{'reason_name': 'high memory usage', 'anomaly_count_for_reason': 97, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 82, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Runtime_7778_JVM_JVM_Uptime', 'JVM-Threads_7778_JVM_ThreadCount_Threads']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 22, 'earliest_anomaly_timestamp_for_reason': 1616491980, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 14, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493420, 'sample_kpi_names': ['OSLinux-CPU_CPU-0_SingleCpuUtil', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 14, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['JVM-Threads_7778_JVM_ThreadCount_Threads', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite']}]}, {'cmdb_id': 'IG01', 'total_anomalies': 151, 'distinct_anomalous_kpis': 18, 'reasons': [{'reason_name': 'high memory usage', 'anomaly_count_for_reason': 89, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed']}, {'reason_name': 'high JVM CPU load', 'anomaly_count_for_reason': 80, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Runtime_7778_JVM_JVM_Uptime', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 19, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493480, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 13, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 10, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493240, 'sample_kpi_names': ['JVM-Operating System_7778_JVM_JVM_CPULoad', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil']}]}, {'cmdb_id': 'Redis01', 'total_anomalies': 97, 'distinct_anomalous_kpis': 11, 'reasons': [{'reason_name': 'high memory usage', 'anomaly_count_for_reason': 30, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 16, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493420, 'sample_kpi_names': ['OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite']}, {'reason_name': 'network latency', 'anomaly_count_for_reason': 16, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 11, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493420, 'sample_kpi_names': ['OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil']}]}, {'cmdb_id': 'apache02', 'total_anomalies': 70, 'distinct_anomalous_kpis': 11, 'reasons': [{'reason_name': 'high memory usage', 'anomaly_count_for_reason': 32, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 13, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 6, 'earliest_anomaly_timestamp_for_reason': 1616492340, 'latest_anomaly_timestamp_for_reason': 1616493540, 'sample_kpi_names': ['OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite']}]}, {'cmdb_id': 'apache01', 'total_anomalies': 64, 'distinct_anomalous_kpis': 11, 'reasons': [{'reason_name': 'high memory usage', 'anomaly_count_for_reason': 22, 'earliest_anomaly_timestamp_for_reason': 1616491860, 'latest_anomaly_timestamp_for_reason': 1616493600, 'sample_kpi_names': ['OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc']}, {'reason_name': 'high CPU usage', 'anomaly_count_for_reason': 17, 'earliest_anomaly_timestamp_for_reason': 1616491800, 'latest_anomaly_timestamp_for_reason': 1616493480, 'sample_kpi_names': ['OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuUtil']}, {'reason_name': 'high disk I/O read usage', 'anomaly_count_for_reason': 5, 'earliest_anomaly_timestamp_for_reason': 1616492100, 'latest_anomaly_timestamp_for_reason': 1616493180, 'sample_kpi_names': ['OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite']}]}], 'JVM_OOM_candidates': []}, Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')  # UTC+8

# Reuse trace_df if present, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global thresholds (count, p95, p5, mean) across full trace series per (cmdb_id, trace_name)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(
        count='count',
        p95=lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan,
        p5 =lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan,
        mean=lambda x: float(x.mean()) if x.notna().any() else np.nan
    )
)

# Drop pairs where p95 or p5 is NaN (cannot evaluate anomalies)
trace_thresholds = trace_thresholds.dropna(subset=['p95','p5']).copy()

# 2) Filter trace.csv to incident window and find anomalous rows
start_ts = 1616491800
end_ts   = 1616493600

trace_window = trace_df[
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Merge with thresholds to evaluate only known (cmdb_id, trace_name) pairs
merged = pd.merge(
    trace_window,
    trace_thresholds[['cmdb_id','trace_name','p95','p5']],
    on=['cmdb_id','trace_name'],
    how='inner'
)

# Identify anomalies: value >= global_p95 OR value <= global_p5
merged['is_anomaly'] = ((merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])) & merged['value'].notna()

anomalies = merged[merged['is_anomaly']].copy()

# Aggregate per (cmdb_id, trace_name)
if anomalies.empty:
    total_anomalous_pairs = 0
    top20_list = []
    top20_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'latest_anomaly_timestamp','max_value_in_window','min_value_in_window',
        'global_p95','global_p5'
    ])
else:
    agg = (
        anomalies
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly','sum'),
            earliest_anomaly_timestamp=('timestamp','min'),
            latest_anomaly_timestamp=('timestamp','max'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            global_p95=('p95','first'),
            global_p5 =('p5','first')
        )
    )
    total_anomalous_pairs = int(agg.shape[0])
    top20_df = agg.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
    
    # Convert to JSON-serializable list of dicts
    def to_native(v):
        if pd.isna(v):
            return None
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (np.floating,)):
            return float(v)
        return v

    top20_list = []
    for _, r in top20_df.iterrows():
        top20_list.append({
            'cmdb_id': r['cmdb_id'],
            'trace_name': r['trace_name'],
            'anomaly_count': to_native(r['anomaly_count']),
            'earliest_anomaly_timestamp': to_native(r['earliest_anomaly_timestamp']),
            'latest_anomaly_timestamp': to_native(r['latest_anomaly_timestamp']),
            'max_value_in_window': to_native(r['max_value_in_window']),
            'min_value_in_window': to_native(r['min_value_in_window']),
            'global_p95': to_native(r['global_p95']),
            'global_p5': to_native(r['global_p5'])
        })

# Assemble compact JSON-serializable report
report = {
    'total_anomalous_pairs': int(total_anomalous_pairs),
    'top20': top20_list
}

# Display compact outputs: report and a concise DataFrame view of top20
report, top20_df[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp','latest_anomaly_timestamp']].head(20)

```
Out[5]:
```
Summary of trace anomalies in window [1616491800, 1616493600]:

- Total distinct (cmdb_id, trace_name) pairs with ≥1 anomaly: 264.

Top 20 anomalous pairs (cmdb_id | trace_name | anomaly_count | earliest_ts | latest_ts | max_in_window | min_in_window | global_p95 | global_p5):

1) dockerA2 | trace.self.duration_p95 | 14 | 1616491800 | 1616493540 | 0.009 | 0.001 | 0.008 | 0.001  
2) dockerA1 | trace.to_MG01.duration_p95 | 6 | 1616492400 | 1616493180 | 1.0154 | 0.008 | 0.96563 | 0.008  
3) dockerA1 | trace.from_MG02.row_count | 6 | 1616491860 | 1616493600 | 79.0 | 1.0 | 69.0 | 10.0  
4) MG01   | trace.from_dockerA1.duration_p95 | 6 | 1616492400 | 1616493180 | 1.0154 | 0.008 | 0.96563 | 0.008  
5) MG02   | trace.from_dockerB1.duration_p95 | 6 | 1616492340 | 1616493360 | 1.01815 | 0.00915 | 1.016 | 0.01  
6) dockerB1 | trace.self.duration_p95 | 6 | 1616492400 | 1616493180 | 0.5417 | 0.009 | 0.10525 | 0.009  
7) dockerB1 | trace.to_MG02.duration_p95 | 6 | 1616492340 | 1616493360 | 1.01815 | 0.00915 | 1.016 | 0.01  
8) MG02   | trace.to_dockerA1.row_count | 6 | 1616491860 | 1616493600 | 79.0 | 1.0 | 69.0 | 10.0  
9) MG01   | trace.from_dockerA2.row_count | 5 | 1616492160 | 1616493060 | 94.0 | 13.0 | 87.2 | 16.0  
10) dockerA2 | trace.to_MG01.row_count | 5 | 1616492160 | 1616493060 | 94.0 | 13.0 | 87.2 | 16.0  
11) dockerA1 | trace.to_MG02.row_count | 5 | 1616491800 | 1616493120 | 35.0 | 2.0 | 24.0 | 2.45  
12) MG02   | trace.from_dockerB1.row_count | 5 | 1616491800 | 1616493600 | 432.0 | 3.0 | 345.5 | 20.0  
13) MG02   | trace.from_dockerA1.row_count | 5 | 1616491800 | 1616493120 | 35.0 | 2.0 | 24.0 | 2.45  
14) MG02   | trace.from_Tomcat01.row_count | 5 | 1616491800 | 1616493600 | 199.0 | 3.0 | 192.5 | 33.0  
15) dockerB1 | trace.to_MG02.row_count | 5 | 1616491800 | 1616493600 | 432.0 | 3.0 | 345.5 | 20.0  
16) dockerB2 | trace.from_MG02.row_count | 5 | 1616491800 | 1616492640 | 110.0 | 16.0 | 100.95 | 17.0  
17) Tomcat01 | trace.to_MG02.row_count | 5 | 1616491800 | 1616493600 | 199.0 | 3.0 | 192.5 | 33.0  
18) MG02   | trace.from_dockerA2.duration_p95 | 5 | 1616492400 | 1616493540 | 1.021 | 0.009 | 1.0196 | 0.0103  
19) MG02   | trace.to_dockerB2.row_count | 5 | 1616491800 | 1616492640 | 110.0 | 16.0 | 100.95 | 17.0  
20) dockerA2 | trace.to_MG02.duration_p95 | 5 | 1616492400 | 1616493540 | 1.021 | 0.009 | 1.0196 | 0.0103

If you want the detailed JSON of these top-20 entries or the full list of all 264 anomalous pairs, I can provide that next.

The original code execution output of IPython Kernel is also provided below for reference:

({'total_anomalous_pairs': 264, 'top20': [{'cmdb_id': 'dockerA2', 'trace_name': 'trace.self.duration_p95', 'anomaly_count': 14, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493540, 'max_value_in_window': 0.009, 'min_value_in_window': 0.001, 'global_p95': 0.008, 'global_p5': 0.001}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.to_MG01.duration_p95', 'anomaly_count': 6, 'earliest_anomaly_timestamp': 1616492400, 'latest_anomaly_timestamp': 1616493180, 'max_value_in_window': 1.0154, 'min_value_in_window': 0.008, 'global_p95': 0.9656299999999998, 'global_p5': 0.008}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.from_MG02.row_count', 'anomaly_count': 6, 'earliest_anomaly_timestamp': 1616491860, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 79.0, 'min_value_in_window': 1.0, 'global_p95': 69.0, 'global_p5': 10.0}, {'cmdb_id': 'MG01', 'trace_name': 'trace.from_dockerA1.duration_p95', 'anomaly_count': 6, 'earliest_anomaly_timestamp': 1616492400, 'latest_anomaly_timestamp': 1616493180, 'max_value_in_window': 1.0154, 'min_value_in_window': 0.008, 'global_p95': 0.9656299999999998, 'global_p5': 0.008}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_dockerB1.duration_p95', 'anomaly_count': 6, 'earliest_anomaly_timestamp': 1616492340, 'latest_anomaly_timestamp': 1616493360, 'max_value_in_window': 1.01815, 'min_value_in_window': 0.00915, 'global_p95': 1.016, 'global_p5': 0.01}, {'cmdb_id': 'dockerB1', 'trace_name': 'trace.self.duration_p95', 'anomaly_count': 6, 'earliest_anomaly_timestamp': 1616492400, 'latest_anomaly_timestamp': 1616493180, 'max_value_in_window': 0.5417, 'min_value_in_window': 0.009, 'global_p95': 0.10525, 'global_p5': 0.009}, {'cmdb_id': 'dockerB1', 'trace_name': 'trace.to_MG02.duration_p95', 'anomaly_count': 6, 'earliest_anomaly_timestamp': 1616492340, 'latest_anomaly_timestamp': 1616493360, 'max_value_in_window': 1.01815, 'min_value_in_window': 0.00915, 'global_p95': 1.016, 'global_p5': 0.01}, {'cmdb_id': 'MG02', 'trace_name': 'trace.to_dockerA1.row_count', 'anomaly_count': 6, 'earliest_anomaly_timestamp': 1616491860, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 79.0, 'min_value_in_window': 1.0, 'global_p95': 69.0, 'global_p5': 10.0}, {'cmdb_id': 'MG01', 'trace_name': 'trace.from_dockerA2.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616492160, 'latest_anomaly_timestamp': 1616493060, 'max_value_in_window': 94.0, 'min_value_in_window': 13.0, 'global_p95': 87.19999999999999, 'global_p5': 16.0}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.to_MG01.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616492160, 'latest_anomaly_timestamp': 1616493060, 'max_value_in_window': 94.0, 'min_value_in_window': 13.0, 'global_p95': 87.19999999999999, 'global_p5': 16.0}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.to_MG02.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493120, 'max_value_in_window': 35.0, 'min_value_in_window': 2.0, 'global_p95': 24.0, 'global_p5': 2.45}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_dockerB1.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 432.0, 'min_value_in_window': 3.0, 'global_p95': 345.5, 'global_p5': 20.0}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_dockerA1.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493120, 'max_value_in_window': 35.0, 'min_value_in_window': 2.0, 'global_p95': 24.0, 'global_p5': 2.45}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_Tomcat01.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 199.0, 'min_value_in_window': 3.0, 'global_p95': 192.5, 'global_p5': 33.0}, {'cmdb_id': 'dockerB1', 'trace_name': 'trace.to_MG02.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 432.0, 'min_value_in_window': 3.0, 'global_p95': 345.5, 'global_p5': 20.0}, {'cmdb_id': 'dockerB2', 'trace_name': 'trace.from_MG02.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616492640, 'max_value_in_window': 110.0, 'min_value_in_window': 16.0, 'global_p95': 100.94999999999997, 'global_p5': 17.0}, {'cmdb_id': 'Tomcat01', 'trace_name': 'trace.to_MG02.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616493600, 'max_value_in_window': 199.0, 'min_value_in_window': 3.0, 'global_p95': 192.5, 'global_p5': 33.0}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_dockerA2.duration_p95', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616492400, 'latest_anomaly_timestamp': 1616493540, 'max_value_in_window': 1.021, 'min_value_in_window': 0.009, 'global_p95': 1.0196, 'global_p5': 0.0103}, {'cmdb_id': 'MG02', 'trace_name': 'trace.to_dockerB2.row_count', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616491800, 'latest_anomaly_timestamp': 1616492640, 'max_value_in_window': 110.0, 'min_value_in_window': 16.0, 'global_p95': 100.94999999999997, 'global_p5': 17.0}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.to_MG02.duration_p95', 'anomaly_count': 5, 'earliest_anomaly_timestamp': 1616492400, 'latest_anomaly_timestamp': 1616493540, 'max_value_in_window': 1.021, 'min_value_in_window': 0.009, 'global_p95': 1.0196, 'global_p5': 0.0103}]},      cmdb_id                        trace_name  anomaly_count  earliest_anomaly_timestamp  latest_anomaly_timestamp
0   dockerA2           trace.self.duration_p95             14                  1616491800                1616493540
1   dockerA1        trace.to_MG01.duration_p95              6                  1616492400                1616493180
2   dockerA1         trace.from_MG02.row_count              6                  1616491860                1616493600
3       MG01  trace.from_dockerA1.duration_p95              6                  1616492400                1616493180
4       MG02  trace.from_dockerB1.duration_p95              6                  1616492340                1616493360
5   dockerB1           trace.self.duration_p95              6                  1616492400                1616493180
6   dockerB1        trace.to_MG02.duration_p95              6                  1616492340                1616493360
7       MG02       trace.to_dockerA1.row_count              6                  1616491860                1616493600
8       MG01     trace.from_dockerA2.row_count              5                  1616492160                1616493060
9   dockerA2           trace.to_MG01.row_count              5                  1616492160                1616493060
10  dockerA1           trace.to_MG02.row_count              5                  1616491800                1616493120
11      MG02     trace.from_dockerB1.row_count              5                  1616491800                1616493600
12      MG02     trace.from_dockerA1.row_count              5                  1616491800                1616493120
13      MG02     trace.from_Tomcat01.row_count              5                  1616491800                1616493600
14  dockerB1           trace.to_MG02.row_count              5                  1616491800                1616493600
15  dockerB2         trace.from_MG02.row_count              5                  1616491800                1616492640
16  Tomcat01           trace.to_MG02.row_count              5                  1616491800                1616493600
17      MG02  trace.from_dockerA2.duration_p95              5                  1616492400                1616493540
18      MG02       trace.to_dockerB2.row_count              5                  1616491800                1616492640
19  dockerA2        trace.to_MG02.duration_p95              5                  1616492400                1616493540)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df and trace_thresholds if present; otherwise load/compute
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

try:
    trace_thresholds
except NameError:
    trace_thresholds = (
        trace_df
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(
            count='count',
            p95=lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan,
            p5 =lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan,
            mean=lambda x: float(x.mean()) if x.notna().any() else np.nan
        )
    )
# drop pairs without valid thresholds
trace_thresholds = trace_thresholds.dropna(subset=['p95','p5']).copy()

# Incident window
start_ts = 1616491800
end_ts   = 1616493600

# Windowed trace rows
trace_window = trace_df[
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Merge with global thresholds (full-series) and identify anomalies
merged = pd.merge(
    trace_window,
    trace_thresholds[['cmdb_id','trace_name','p95','p5']],
    on=['cmdb_id','trace_name'],
    how='inner'
)
merged['is_anomaly'] = ((merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])) & merged['value'].notna()

anomalies = merged[merged['is_anomaly']].copy()

# Allowed root-cause components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# Filter anomalies to allowed components
anomalies_allowed = anomalies[anomalies['cmdb_id'].isin(allowed_components)].copy()

# 1) Per-component summary
if anomalies_allowed.empty:
    per_component_report = []
    per_comp_df = pd.DataFrame(columns=['cmdb_id','total_anomalous_trace_rows','distinct_anomalous_trace_names','earliest_anomaly_timestamp'])
else:
    per_comp = (
        anomalies_allowed
        .groupby('cmdb_id', as_index=False)
        .agg(
            total_anomalous_trace_rows=('is_anomaly','sum'),
            distinct_anomalous_trace_names=('trace_name', lambda x: int(x.nunique())),
            earliest_anomaly_timestamp=('timestamp','min')
        )
    )
    per_comp = per_comp.sort_values('total_anomalous_trace_rows', ascending=False).head(20).reset_index(drop=True)
    # JSON-serializable list
    per_component_report = [
        {
            'cmdb_id': row['cmdb_id'],
            'total_anomalous_trace_rows': int(row['total_anomalous_trace_rows']),
            'distinct_anomalous_trace_names': int(row['distinct_anomalous_trace_names']),
            'earliest_anomaly_timestamp': int(row['earliest_anomaly_timestamp'])
        }
        for _, row in per_comp.iterrows()
    ]
    per_comp_df = per_comp

# 2) Inter-service anomalous trace map (trace_name containing 'to_' or 'from_')
if anomalies_allowed.empty:
    inter_service_report = []
    inter_service_df = pd.DataFrame(columns=['caller_cmdb_id','peer_component','trace_name','anomaly_count','earliest_anomaly_timestamp'])
else:
    mask = anomalies_allowed['trace_name'].str.contains(r'(?:\bto_|\\bto_|(?:from_))', case=False, na=False) \
           | anomalies_allowed['trace_name'].str.contains(r'(?:from_)', case=False, na=False)
    # Simpler: check both substrings case-insensitively
    mask = anomalies_allowed['trace_name'].str.lower().str.contains('to_') | anomalies_allowed['trace_name'].str.lower().str.contains('from_')
    inter = anomalies_allowed[mask].copy()
    if inter.empty:
        inter_service_report = []
        inter_service_df = pd.DataFrame(columns=['caller_cmdb_id','peer_component','trace_name','anomaly_count','earliest_anomaly_timestamp'])
    else:
        # extract peer component using regex (capture alphanum sequence after to_ or from_)
        inter['peer_component'] = inter['trace_name'].str.extract(r'(?:to_|from_)([A-Za-z0-9]+)', flags=re.IGNORECASE, expand=False)
        # drop rows where extraction failed
        inter = inter[inter['peer_component'].notna()].copy()
        # Normalize peer component to original-case as extracted (regex preserves case)
        # Aggregate by caller (cmdb_id), peer_component, trace_name
        inter_agg = (
            inter
            .groupby(['cmdb_id','peer_component','trace_name'], as_index=False)
            .agg(
                anomaly_count=('is_anomaly','sum'),
                earliest_anomaly_timestamp=('timestamp','min')
            )
        )
        inter_agg = inter_agg.sort_values('anomaly_count', ascending=False).head(50).reset_index(drop=True)
        inter_service_report = [
            {
                'caller_cmdb_id': row['cmdb_id'],
                'peer_component': row['peer_component'],
                'trace_name': row['trace_name'],
                'anomaly_count_for_this_pair': int(row['anomaly_count']),
                'earliest_anomaly_timestamp_for_pair': int(row['earliest_anomaly_timestamp'])
            }
            for _, row in inter_agg.iterrows()
        ]
        inter_service_df = inter_agg.rename(columns={'cmdb_id':'caller_cmdb_id','anomaly_count':'anomaly_count'})

# Assemble final JSON-serializable outputs
report = {
    'per_component_summary_top20': per_component_report,
    'inter_service_mappings_top50': inter_service_report
}

# Display compact outputs: report dict, per_comp_df head (top components), inter_service_df head (top mappings)
report, per_comp_df[['cmdb_id','total_anomalous_trace_rows','distinct_anomalous_trace_names','earliest_anomaly_timestamp']].head(20), inter_service_df.head(50)

```
Out[6]:
```
python
summary = (
    "Trace anomaly summary (window 1616491800–1616493600) for allowed root-cause components:\n\n"
    "1) Per-component totals (top by anomalous trace rows):\n"
    "- MG02: 171 anomalous trace rows (42 distinct trace names), earliest anomaly 1616491800\n"
    "- MG01: 165 anomalous trace rows (42 distinct trace names), earliest anomaly 1616491800\n"
    "- IG01: 67 anomalous trace rows (18 distinct trace names), earliest anomaly 1616491800\n"
    "- Tomcat01: 67 anomalous trace rows (18 distinct trace names), earliest anomaly 1616491800\n"
    "- Tomcat04: 67 anomalous trace rows (18 distinct trace names), earliest anomaly 1616491800\n"
    "- IG02: 66 anomalous trace rows (18 distinct trace names), earliest anomaly 1616491800\n"
    "- Tomcat02: 66 anomalous trace rows (18 distinct trace names), earliest anomaly 1616491800\n"
    "- Tomcat03: 66 anomalous trace rows (18 distinct trace names), earliest anomaly 1616491800\n\n"
    "2) Inter-service anomalous trace map (caller -> peer) — top mappings by anomaly count:\n"
    "- MG02 -> dockerA1 (trace.to_dockerA1.row_count): 6 anomalies, earliest 1616491860\n"
    "- MG02 -> dockerB1 (trace.from_dockerB1.duration_p95): 6 anomalies, earliest 1616492340\n"
    "- MG01 -> dockerA1 (trace.from_dockerA1.duration_p95): 6 anomalies, earliest 1616492400\n"
    "- MG02 -> dockerB1 (trace.from_dockerB1.row_count): 5 anomalies, earliest 1616491800\n"
    "- MG01 -> dockerA2 (trace.from_dockerA2.row_count): 5 anomalies, earliest 1616492160\n"
    "- MG02 -> Tomcat01 (trace.from_Tomcat01.row_count): 5 anomalies, earliest 1616491800\n"
    "- MG02 -> dockerA2 (trace.from_dockerA2.duration_p95): 5 anomalies, earliest 1616492400\n"
    "- MG02 -> dockerA1 (trace.from_dockerA1.row_count): 5 anomalies, earliest 1616491800\n"
    "- MG02 -> dockerB2 (trace.to_dockerB2.row_count): 5 anomalies, earliest 1616491800\n"
    "- Tomcat01 -> MG02 (trace.to_MG02.row_count): 5 anomalies, earliest 1616491800\n\n"
    "Notes:\n"
    "- Only trace anomalies for the allowed components were considered.\n"
    "- Earliest anomalies are mostly at the start of the window (1616491800).\n"
    "- Inter-service mappings show several MG01/MG02 related caller->peer traces with repeated anomalous counts (top counts are 6 or 5).\n"
    "- If you want the full per-component list or the full set of caller->peer mappings (up to 50 entries), I can provide the JSON structure produced earlier."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

({'per_component_summary_top20': [{'cmdb_id': 'MG02', 'total_anomalous_trace_rows': 171, 'distinct_anomalous_trace_names': 42, 'earliest_anomaly_timestamp': 1616491800}, {'cmdb_id': 'MG01', 'total_anomalous_trace_rows': 165, 'distinct_anomalous_trace_names': 42, 'earliest_anomaly_timestamp': 1616491800}, {'cmdb_id': 'IG01', 'total_anomalous_trace_rows': 67, 'distinct_anomalous_trace_names': 18, 'earliest_anomaly_timestamp': 1616491800}, {'cmdb_id': 'Tomcat01', 'total_anomalous_trace_rows': 67, 'distinct_anomalous_trace_names': 18, 'earliest_anomaly_timestamp': 1616491800}, {'cmdb_id': 'Tomcat04', 'total_anomalous_trace_rows': 67, 'distinct_anomalous_trace_names': 18, 'earliest_anomaly_timestamp': 1616491800}, {'cmdb_id': 'IG02', 'total_anomalous_trace_rows': 66, 'distinct_anomalous_trace_names': 18, 'earliest_anomaly_timestamp': 1616491800}, {'cmdb_id': 'Tomcat02', 'total_anomalous_trace_rows': 66, 'distinct_anomalous_trace_names': 18, 'earliest_anomaly_timestamp': 1616491800}, {'cmdb_id': 'Tomcat03', 'total_anomalous_trace_rows': 66, 'distinct_anomalous_trace_names': 18, 'earliest_anomaly_timestamp': 1616491800}], 'inter_service_mappings_top50': [{'caller_cmdb_id': 'MG02', 'peer_component': 'dockerA1', 'trace_name': 'trace.to_dockerA1.row_count', 'anomaly_count_for_this_pair': 6, 'earliest_anomaly_timestamp_for_pair': 1616491860}, {'caller_cmdb_id': 'MG02', 'peer_component': 'dockerB1', 'trace_name': 'trace.from_dockerB1.duration_p95', 'anomaly_count_for_this_pair': 6, 'earliest_anomaly_timestamp_for_pair': 1616492340}, {'caller_cmdb_id': 'MG01', 'peer_component': 'dockerA1', 'trace_name': 'trace.from_dockerA1.duration_p95', 'anomaly_count_for_this_pair': 6, 'earliest_anomaly_timestamp_for_pair': 1616492400}, {'caller_cmdb_id': 'MG02', 'peer_component': 'dockerB1', 'trace_name': 'trace.from_dockerB1.row_count', 'anomaly_count_for_this_pair': 5, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'MG01', 'peer_component': 'dockerA2', 'trace_name': 'trace.from_dockerA2.row_count', 'anomaly_count_for_this_pair': 5, 'earliest_anomaly_timestamp_for_pair': 1616492160}, {'caller_cmdb_id': 'MG02', 'peer_component': 'Tomcat01', 'trace_name': 'trace.from_Tomcat01.row_count', 'anomaly_count_for_this_pair': 5, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'MG02', 'peer_component': 'dockerA2', 'trace_name': 'trace.from_dockerA2.duration_p95', 'anomaly_count_for_this_pair': 5, 'earliest_anomaly_timestamp_for_pair': 1616492400}, {'caller_cmdb_id': 'MG02', 'peer_component': 'dockerA1', 'trace_name': 'trace.from_dockerA1.row_count', 'anomaly_count_for_this_pair': 5, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'MG02', 'peer_component': 'dockerB2', 'trace_name': 'trace.to_dockerB2.row_count', 'anomaly_count_for_this_pair': 5, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'Tomcat01', 'peer_component': 'MG02', 'trace_name': 'trace.to_MG02.row_count', 'anomaly_count_for_this_pair': 5, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat03', 'trace_name': 'trace.to_Tomcat03.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat04', 'trace_name': 'trace.to_Tomcat04.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat01', 'trace_name': 'trace.to_Tomcat01.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat02', 'trace_name': 'trace.to_Tomcat02.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat04', 'trace_name': 'trace.to_Tomcat04.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat04', 'trace_name': 'trace.to_Tomcat04.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat02', 'trace_name': 'trace.to_Tomcat02.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat03', 'trace_name': 'trace.to_Tomcat03.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat03', 'trace_name': 'trace.to_Tomcat03.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat02', 'trace_name': 'trace.to_Tomcat02.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat03', 'trace_name': 'trace.to_Tomcat03.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat04', 'trace_name': 'trace.to_Tomcat04.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat04', 'trace_name': 'trace.to_Tomcat04.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492580}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat04', 'trace_name': 'trace.to_Tomcat04.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat03', 'trace_name': 'trace.from_Tomcat03.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat03', 'trace_name': 'trace.from_Tomcat03.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat03', 'trace_name': 'trace.from_Tomcat03.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492220}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat01', 'trace_name': 'trace.from_Tomcat01.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat01', 'trace_name': 'trace.from_Tomcat01.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492340}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat02', 'trace_name': 'trace.from_Tomcat02.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat01', 'trace_name': 'trace.to_Tomcat01.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG02', 'peer_component': 'Tomcat01', 'trace_name': 'trace.to_Tomcat01.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491860}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat01', 'trace_name': 'trace.to_Tomcat01.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat01', 'trace_name': 'trace.to_Tomcat01.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat01', 'trace_name': 'trace.to_Tomcat01.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat02', 'trace_name': 'trace.to_Tomcat02.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492700}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat02', 'trace_name': 'trace.to_Tomcat02.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492520}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat02', 'trace_name': 'trace.to_Tomcat02.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491860}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat03', 'trace_name': 'trace.to_Tomcat03.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492520}, {'caller_cmdb_id': 'IG01', 'peer_component': 'Tomcat03', 'trace_name': 'trace.to_Tomcat03.duration_p95', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492520}, {'caller_cmdb_id': 'Tomcat04', 'peer_component': 'MG02', 'trace_name': 'trace.to_MG02.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'Tomcat04', 'peer_component': 'MG01', 'trace_name': 'trace.to_MG01.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492700}, {'caller_cmdb_id': 'Tomcat04', 'peer_component': 'MG02', 'trace_name': 'trace.to_MG02.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492400}, {'caller_cmdb_id': 'Tomcat01', 'peer_component': 'IG01', 'trace_name': 'trace.from_IG01.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'Tomcat01', 'peer_component': 'IG02', 'trace_name': 'trace.from_IG02.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}, {'caller_cmdb_id': 'Tomcat01', 'peer_component': 'MG01', 'trace_name': 'trace.to_MG01.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492340}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat02', 'trace_name': 'trace.from_Tomcat02.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492040}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat04', 'trace_name': 'trace.from_Tomcat04.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616493180}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat01', 'trace_name': 'trace.from_Tomcat01.row_count', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616492640}, {'caller_cmdb_id': 'MG01', 'peer_component': 'Tomcat02', 'trace_name': 'trace.from_Tomcat02.duration_mean', 'anomaly_count_for_this_pair': 4, 'earliest_anomaly_timestamp_for_pair': 1616491800}]},     cmdb_id  total_anomalous_trace_rows  distinct_anomalous_trace_names  earliest_anomaly_timestamp
0      MG02                         171                              42                  1616491800
1      MG01                         165                              42                  1616491800
2      IG01                          67                              18                  1616491800
3  Tomcat01                          67                              18                  1616491800
4  Tomcat04                          67                              18                  1616491800
5      IG02                          66                              18                  1616491800
6  Tomcat02                          66                              18                  1616491800
7  Tomcat03                          66                              18                  1616491800,    caller_cmdb_id peer_component                         trace_name  anomaly_count  earliest_anomaly_timestamp
0            MG02       dockerA1        trace.to_dockerA1.row_count              6                  1616491860
1            MG02       dockerB1   trace.from_dockerB1.duration_p95              6                  1616492340
2            MG01       dockerA1   trace.from_dockerA1.duration_p95              6                  1616492400
3            MG02       dockerB1      trace.from_dockerB1.row_count              5                  1616491800
4            MG01       dockerA2      trace.from_dockerA2.row_count              5                  1616492160
..            ...            ...                                ...            ...                         ...
45       Tomcat01           MG01        trace.to_MG01.duration_mean              4                  1616492340
46           MG01       Tomcat02      trace.from_Tomcat02.row_count              4                  1616492040
47           MG01       Tomcat04  trace.from_Tomcat04.duration_mean              4                  1616493180
48           MG01       Tomcat01      trace.from_Tomcat01.row_count              4                  1616492640
49           MG01       Tomcat02  trace.from_Tomcat02.duration_mean              4                  1616491800

[50 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse log_df if present, else load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure numeric value
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Allowed components and incident window
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]
start_ts = 1616491800
end_ts   = 1616493600

# Filter logs to allowed components and window
logs_win = log_df[
    (log_df['cmdb_id'].isin(allowed_components)) &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

# Aggregate total_error_count (sum of value where log_name=='log.error_count')
err_mask = logs_win['log_name'] == 'log.error_count'
row_mask = logs_win['log_name'] == 'log.row_count'

total_error = (
    logs_win[err_mask]
    .groupby('cmdb_id', as_index=False)['value']
    .sum()
    .rename(columns={'value':'total_error_count'})
)

total_row = (
    logs_win[row_mask]
    .groupby('cmdb_id', as_index=False)['value']
    .sum()
    .rename(columns={'value':'total_row_count'})
)

# earliest_error_ts: min timestamp where log.error_count and value>0
err_pos = logs_win[err_mask & (logs_win['value'] > 0)].copy()
earliest_err_ts = (
    err_pos.groupby('cmdb_id', as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_error_ts'})
)

# earliest_low_rowcount_ts: min timestamp where log.row_count and value == 0
row_zero = logs_win[row_mask & (logs_win['value'] == 0)].copy()
earliest_zero_ts = (
    row_zero.groupby('cmdb_id', as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_low_rowcount_ts'})
)

# Merge all summaries
summary_df = pd.DataFrame({'cmdb_id': sorted(allowed_components)})
summary_df = summary_df.merge(total_error, on='cmdb_id', how='left')
summary_df = summary_df.merge(total_row, on='cmdb_id', how='left')
summary_df = summary_df.merge(earliest_err_ts, on='cmdb_id', how='left')
summary_df = summary_df.merge(earliest_zero_ts, on='cmdb_id', how='left')

# Fill NaNs for counts with 0
summary_df['total_error_count'] = summary_df['total_error_count'].fillna(0)
summary_df['total_row_count'] = summary_df['total_row_count'].fillna(0)

# Keep only components that have non-zero total_error_count OR have a zero row_count timestamp
mask_issue = (summary_df['total_error_count'] != 0) | (summary_df['earliest_low_rowcount_ts'].notna())
issues_df = summary_df[mask_issue].copy()

# Sort by total_error_count desc and limit to top 20
issues_df = issues_df.sort_values('total_error_count', ascending=False).head(20).reset_index(drop=True)

# Prepare JSON-serializable list
def to_native(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        # if integer-like float, cast to int to keep compactness
        if float(v).is_integer():
            return int(v)
        return float(v)
    return v

result_list = []
for _, r in issues_df.iterrows():
    result_list.append({
        'cmdb_id': r['cmdb_id'],
        'total_error_count': to_native(r['total_error_count']),
        'total_row_count': to_native(r['total_row_count']),
        'earliest_error_ts': to_native(r['earliest_error_ts']),
        'earliest_low_rowcount_ts': to_native(r['earliest_low_rowcount_ts'])
    })

# Compact DataFrame view for controller
compact_view = issues_df[['cmdb_id','total_error_count','total_row_count','earliest_error_ts','earliest_low_rowcount_ts']].head(20)

# Display results (JSON-serializable list and compact DataFrame)
result_list, compact_view

```
Out[7]:
```
summary = (
    "Log analysis (window 1616491800–1616493600) for allowed components:\n"
    "- No components showed log-level issues. For every allowed component, the summed log.error_count is 0 and there were no timestamps with log.row_count == 0 in the window.\n"
    "- Therefore the returned list of components with log issues is empty."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

([], Empty DataFrame
Columns: [cmdb_id, total_error_count, total_row_count, earliest_error_ts, earliest_low_rowcount_ts]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_df and thresholds from kernel state; reload if missing
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    thresholds
except NameError:
    # compute global thresholds per rule 9 if not present
    metric_allowed_full = metric_df.copy()
    thresholds = (
        metric_allowed_full
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .agg(
            count='count',
            p95=lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan,
            p5 =lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan,
            mean=lambda x: float(x.mean()) if x.notna().any() else np.nan
        )
    )
thresholds = thresholds.dropna(subset=['p95','p5']).copy()

# Allowed root-cause components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# Incident window
start_ts = 1616491800
end_ts   = 1616493600

# Filter metric rows to window and allowed components
metric_window = metric_df[
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['cmdb_id'].isin(allowed_components))
].copy()

# Merge with global thresholds to evaluate anomalies
metric_merged = pd.merge(
    metric_window,
    thresholds[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='inner'
)

# Identify anomalous rows
metric_merged['is_anomaly'] = ((metric_merged['value'] >= metric_merged['p95']) | (metric_merged['value'] <= metric_merged['p5'])) & metric_merged['value'].notna()
anomalous_rows = metric_merged[metric_merged['is_anomaly']].copy()

# If no anomalies, return empty list
if anomalous_rows.empty:
    component_severity_report = []
    component_severity_report, pd.DataFrame(component_severity_report).head(20)
else:
    # Compute severity per row
    def compute_sev(row):
        v = row['value']
        p95 = row['p95']
        p5 = row['p5']
        if pd.isna(v):
            return 0.0
        if v >= p95:
            denom = abs(p95) if p95 != 0 else 1.0
            return float((v - p95) / denom)
        elif v <= p5:
            denom = abs(p5) if p5 != 0 else 1.0
            return float((p5 - v) / denom)
        else:
            return 0.0

    anomalous_rows['severity'] = anomalous_rows.apply(compute_sev, axis=1)

    # Reason/category mapping function (case-insensitive); returns list of categories (may be multiple)
    def map_reasons(kpi):
        k = (kpi or "").lower()
        reasons = []
        # memory/heap/swap
        if any(x in k for x in ['memory','heap','heapmemory','heapmemoryused','heapmemorymax','swap']):
            reasons.append('high memory usage')
        # cpu/load
        if ('cpu' in k) or ('load' in k):
            reasons.append('high CPU usage')
        # network
        if any(x in k for x in ['network','tcp','udp','packet']):
            reasons.append('network latency')
            if 'packet' in k:
                reasons.append('network packet loss')
        # disk/io/read/write/filesystem/df
        if any(x in k for x in [' io',' io_',' io-','io','read','write','dskread','dskreadwrite','diskio','localdisk']):
            # prefer I/O read usage if 'io' or 'read' or 'dsk' present
            reasons.append('high disk I/O read usage')
        if any(x in k for x in ['fs','fscapacity','fsused','fsavailable','filesystem',' df','inode']):
            reasons.append('high disk space usage')
        # jvm/gc/thread
        if any(x in k for x in ['jvm','gc','gc_pause','thread','threadcount']):
            # classify as high JVM CPU load
            reasons.append('high JVM CPU load')
        # ensure unique
        return list(dict.fromkeys(reasons))

    anomalous_rows['mapped_reasons'] = anomalous_rows['kpi_name'].apply(map_reasons)

    # Determine threshold used for each anomalous row
    def which_threshold(row):
        v = row['value']
        if v >= row['p95']:
            return 'p95'
        elif v <= row['p5']:
            return 'p5'
        return None
    anomalous_rows['threshold_used'] = anomalous_rows.apply(which_threshold, axis=1)

    # For JVM-Heap pair candidate existence per component (full series, per rule 9)
    metric_df['kpi_lower'] = metric_df['kpi_name'].str.lower()
    heap_pair_flags = {}
    for comp in allowed_components:
        sub = metric_df[metric_df['cmdb_id']==comp]
        has_used = sub['kpi_lower'].str.contains('heapmemoryused', na=False).any()
        has_max  = sub['kpi_lower'].str.contains('heapmemorymax', na=False).any()
        heap_pair_flags[comp] = bool(has_used and has_max)

    # For each component, find the anomalous row with max severity
    grouped_max = (
        anomalous_rows
        .sort_values(['cmdb_id','severity','timestamp'], ascending=[True, False, True])
        .groupby('cmdb_id', as_index=False)
        .first()  # first after sorting by severity desc gives max severity row per cmdb_id
    )

    # Prepare final report DataFrame
    report_df = pd.DataFrame({
        'component': grouped_max['cmdb_id'],
        'max_severity': grouped_max['severity'],
        'reason_category_of_max': grouped_max['mapped_reasons'].apply(lambda x: x if isinstance(x, list) else []),
        'kpi_name_of_max': grouped_max['kpi_name'],
        'timestamp_of_max_severity': grouped_max['timestamp'],
        'value_of_max': grouped_max['value'],
        'threshold_used': grouped_max['threshold_used'],
        'jvm_heap_pair_candidate': grouped_max['cmdb_id'].map(heap_pair_flags)
    })

    # Sort by max_severity desc and limit to top 20
    report_df = report_df.sort_values('max_severity', ascending=False).head(20).reset_index(drop=True)

    # Convert types to JSON-serializable native types and compact mapped reasons into comma-separated string
    def native_row(r):
        return {
            'component': r['component'],
            'max_severity': float(r['max_severity']) if not pd.isna(r['max_severity']) else None,
            'reason_category_of_max': r['reason_category_of_max'] if isinstance(r['reason_category_of_max'], list) else [],
            'kpi_name_of_max': r['kpi_name_of_max'],
            'timestamp_of_max_severity': int(r['timestamp_of_max_severity']) if not pd.isna(r['timestamp_of_max_severity']) else None,
            'value_of_max': float(r['value_of_max']) if not pd.isna(r['value_of_max']) else None,
            'threshold_used': r['threshold_used'],
            'jvm_heap_pair_candidate': bool(r['jvm_heap_pair_candidate'])
        }

    component_severity_report = [native_row(r) for _, r in report_df.iterrows()]

    # Display compact DataFrame view as well
    compact_view = report_df[['component','max_severity','kpi_name_of_max','timestamp_of_max_severity','value_of_max','threshold_used','jvm_heap_pair_candidate']].head(20)

    component_severity_report, compact_view

```
Out[8]:
```
I ran the severity computation cell but it returned None and no JSON report was produced. This indicates the cell did not produce the expected component severity results (likely because no anomalous rows were yielded or the final values were not returned).

Suggested next steps:
- I can re-run the computation and explicitly display:
  - anomalous_rows.shape to confirm whether anomalous metric rows exist in the window, and
  - a small sample anomalous_rows.head() and the computed component_severity_report.
- Or I can immediately re-run the last analysis cell and return the JSON list of component severity records.

Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df and thresholds if present; otherwise load/compute (global thresholds must be computed on full series)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    thresholds
except NameError:
    thresholds = (
        metric_df
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(
            count='count',
            p95=lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan,
            p5 =lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan,
            mean=lambda x: float(x.mean()) if x.notna().any() else np.nan
        )
    )
thresholds = thresholds.dropna(subset=['p95','p5']).copy()

# Allowed components and incident window
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]
start_ts = 1616491800
end_ts   = 1616493600

# Filter metric rows to window and allowed components
metric_window = metric_df[
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['cmdb_id'].isin(allowed_components))
].copy()

# Merge with global thresholds (full-series) and identify anomalies
metric_merged = pd.merge(
    metric_window,
    thresholds[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='inner'
)

metric_merged['is_anomaly'] = ((metric_merged['value'] >= metric_merged['p95']) | (metric_merged['value'] <= metric_merged['p5'])) & metric_merged['value'].notna()
anomalous_rows = metric_merged[metric_merged['is_anomaly']].copy()

# 1) anomalous_rows.shape
anomalous_shape = anomalous_rows.shape

# If empty, return shape and explanatory message
if anomalous_rows.empty:
    message = "No anomalous metric rows found in the window for allowed components."
    anomalous_shape, message
else:
    # 2) small sample with breach_direction
    def breach_dir(r):
        if r['value'] >= r['p95']:
            return 'upper'
        elif r['value'] <= r['p5']:
            return 'lower'
        return None

    sample_df = anomalous_rows[['timestamp','cmdb_id','kpi_name','value','p95','p5']].head(10).copy()
    sample_df = sample_df.rename(columns={'p95':'global_p95','p5':'global_p5'})
    sample_df['breach_direction'] = sample_df.apply(lambda r: 'upper' if r['value'] >= r['global_p95'] else ('lower' if r['value'] <= r['global_p5'] else None), axis=1)

    # 3) compute severity per anomalous row
    def compute_severity(v, p95, p5):
        if pd.isna(v):
            return 0.0
        if v >= p95:
            denom = abs(p95) if p95 != 0 else 1.0
            return float((v - p95) / denom)
        elif v <= p5:
            denom = abs(p5) if p5 != 0 else 1.0
            return float((p5 - v) / denom)
        else:
            return 0.0

    anomalous_rows['severity'] = anomalous_rows.apply(lambda r: compute_severity(r['value'], r['p95'], r['p5']), axis=1)

    # reason/category mapping (returns list)
    def map_reasons(kpi):
        k = (kpi or "").lower()
        reasons = []
        # memory / heap / swap
        if any(x in k for x in ['memory','heap','heapmemory','heapmemoryused','heapmemorymax','swap']):
            reasons.append('high memory usage')
        # CPU / load
        if ('cpu' in k) or ('load' in k):
            reasons.append('high CPU usage')
        # network
        if any(x in k for x in ['network','tcp','udp']):
            reasons.append('network latency')
        if 'packet' in k:
            reasons.append('network packet loss')
        # disk / io / read / write / fs
        if any(x in k for x in [' io',' io_',' io-','io','read','write','dskread','dskreadwrite','diskio','localdisk']):
            reasons.append('high disk I/O read usage')
        if any(x in k for x in ['fs','fscapacity','fsused','fsavailable','filesystem',' df','inode']):
            reasons.append('high disk space usage')
        # jvm/gc/thread
        if any(x in k for x in ['jvm','gc','gc_pause','thread','threadcount']):
            reasons.append('high JVM CPU load')
        # unique preserve order
        return list(dict.fromkeys(reasons))

    anomalous_rows['mapped_reasons'] = anomalous_rows['kpi_name'].apply(map_reasons)

    # determine threshold used
    anomalous_rows['threshold_used'] = anomalous_rows.apply(lambda r: 'p95' if r['value'] >= r['p95'] else ('p5' if r['value'] <= r['p5'] else None), axis=1)

    # jvm_heap_pair_exists per component using full series
    metric_df['kpi_lower'] = metric_df['kpi_name'].str.lower()
    heap_pair_flags = {}
    for comp in allowed_components:
        sub = metric_df[metric_df['cmdb_id']==comp]
        has_used = sub['kpi_lower'].str.contains('heapmemoryused', na=False).any()
        has_max  = sub['kpi_lower'].str.contains('heapmemorymax', na=False).any()
        heap_pair_flags[comp] = bool(has_used and has_max)

    # For each component find row with max severity (tie-breaker: earliest timestamp)
    anomalous_rows_sorted = anomalous_rows.sort_values(['cmdb_id','severity','timestamp'], ascending=[True, False, True])
    max_per_comp = anomalous_rows_sorted.groupby('cmdb_id', as_index=False).first()

    # Build report DataFrame for allowed components (only those present in max_per_comp)
    report_df = pd.DataFrame({
        'component': max_per_comp['cmdb_id'],
        'max_severity': max_per_comp['severity'],
        'reason_category_of_max': max_per_comp['mapped_reasons'],
        'kpi_name_of_max': max_per_comp['kpi_name'],
        'timestamp_of_max_severity': max_per_comp['timestamp'],
        'value_of_max': max_per_comp['value'],
        'threshold_used': max_per_comp['threshold_used'],
        'jvm_heap_pair_exists': max_per_comp['cmdb_id'].map(heap_pair_flags)
    })

    report_df = report_df.sort_values('max_severity', ascending=False).head(20).reset_index(drop=True)

    # JSON-serializable list
    def native_row(r):
        return {
            'component': r['component'],
            'max_severity': float(r['max_severity']) if not pd.isna(r['max_severity']) else None,
            'reason_category_of_max': r['reason_category_of_max'] if isinstance(r['reason_category_of_max'], list) else [],
            'kpi_name_of_max': r['kpi_name_of_max'],
            'timestamp_of_max_severity': int(r['timestamp_of_max_severity']) if not pd.isna(r['timestamp_of_max_severity']) else None,
            'value_of_max': float(r['value_of_max']) if not pd.isna(r['value_of_max']) else None,
            'threshold_used': r['threshold_used'],
            'jvm_heap_pair_exists': bool(r['jvm_heap_pair_exists'])
        }

    component_severity_list = [native_row(r) for _, r in report_df.iterrows()]

    # Return compact bounded outputs: shape, sample (head 10), component severity list (top 20)
    anomalous_shape, sample_df[['timestamp','cmdb_id','kpi_name','value','global_p95','global_p5','breach_direction']], component_severity_list

```
Out[9]:
```
The re-run executed but produced no returned outputs (the computation cell completed and returned None), so I do not have the requested anomalous_rows.shape, sample rows, or component severity list available from that run.

Likely cause: the cell did not explicitly return the variables at the end. I can re-run the computation now and explicitly return the three bounded outputs you requested (1) anomalous_rows.shape, (2) sample anomalous rows (up to 10) with the specified columns and breach_direction, and (3) the component severity list (top 20) as JSON-serializable objects.

Would you like me to re-run and return those results now?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```